In [1]:
import pandas as pd
import requests
import json
import os
import time

print("Pandas:", pd.__version__)
print("Requests:", requests.__version__)
print("Setup OK")

Pandas: 2.3.3
Requests: 2.32.5
Setup OK


In [2]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "gemma3:1b"

response = requests.get("http://localhost:11434/api/tags")

print("Status:", response.status_code)

if response.status_code == 200:
    print("Ollama connection: OK")
    print("\nInstalled models:")

    for model in response.json().get("models", []):
        print("-", model["name"])
else:
    print("Ollama connection FAILED")

Status: 200
Ollama connection: OK

Installed models:
- gemma3:1b


In [3]:
folder = r"C:\Users\LENOVO\OneDrive - London Metropolitan University\Documents\hexagon-research"

csv_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith(".csv")
]

print("CSV files found:")

for i, file in enumerate(csv_files):
    print(i, "->", file)

CSV files found:
0 -> Hexagon-Dataset - Culture+Religion+Literature.csv


In [6]:
csv_files = [
    f for f in os.listdir(folder)
    if f.lower().endswith(".csv")
]

print("CSV files found:")

for i, file in enumerate(csv_files):
    print(i, "->", file)

CSV files found:
0 -> Hexagon-Dataset - Culture+Religion+Literature.csv


In [7]:
file_name = csv_files[0]
file_path = os.path.join(folder, file_name)

df = pd.read_csv(file_path)

print("File:", file_name)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

display(df.head(2))

File: Hexagon-Dataset - Culture+Religion+Literature.csv
Rows: 424
Columns: 18

Columns:
['Question ID', 'Domain', 'Subdomain', 'Question Type', 'Difficulty', 'English Question', 'Nepali Question', 'Gold Answer (EN)', 'Gold Answer (NE)', 'Explanation', 'Source ID', 'Source Title', 'Source URL', 'Source Type', 'Created By', 'Reviewed By', 'Status', 'Notes']


,Question ID,Domain,Subdomain,Question Type,Difficulty,English Question,Nepali Question,Gold Answer (EN),Gold Answer (NE),Explanation,Source ID,Source Title,Source URL,Source Type,Created By,Reviewed By,Status,Notes
0,CUL001,Culture,Festivals,Fact Retrieval,Easy,For how many days is the Dashain festival trad...,नेपालमा दशैं पर्व परम्परागत रूपमा कति दिनसम्म ...,15,15,"Dashain runs 15 days, from Ghatasthapana to Ko...",S01,"Ministry of Culture, Tourism and Civil Aviation",https://www.tourism.gov.np,Government,Raunak Sharma,NaN,done,verified fact and citation of listed source.
1,CUL002,Culture,Festivals,Definition,Easy,What is the name of the day during Tihar when ...,तिहारमा कुकुरको पूजा गरिने दिनलाई के भनिन्छ?,Kukur Tihar,कुकुर तिहार,The second day of Tihar (Yamapanchak) is dedic...,S01,"Ministry of Culture, Tourism and Civil Aviation",https://www.tourism.gov.np,Government,Raunak Sharma,NaN,done,verified fact and citation of listed source.


In [8]:
required_columns = [
    "Question ID",
    "English Question",
    "Nepali Question",
    "Gold Answer (EN)",
    "Gold Answer (NE)"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    print("❌ Missing columns:", missing)
else:
    print("✅ All required columns exist")

✅ All required columns exist


In [9]:
dataset_context = df.to_csv(index=False)

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))
print("Dataset characters:", len(dataset_context))
print("Approximate tokens:", len(dataset_context) // 4)

Number of rows: 424
Number of columns: 18
Dataset characters: 178739
Approximate tokens: 44684


In [ ]:
CHUNK_SIZE = 150

all_results = []

num_chunks = (len(df) + CHUNK_SIZE - 1) // CHUNK_SIZE

print("Total questions:", len(df))
print("Chunk size:", CHUNK_SIZE)
print("Number of chunks:", num_chunks)

for chunk_no, start in enumerate(
    range(0, len(df), CHUNK_SIZE),
    start=1
):

    chunk = df.iloc[start:start + CHUNK_SIZE].copy()
    chunk_context = chunk.to_csv(index=False)

    print("\n" + "=" * 70)
    print(f"CHUNK {chunk_no}/{num_chunks}")
    print(f"Rows {start + 1} to {start + len(chunk)}")
    print("Context characters:", len(chunk_context))
    print("Approximate tokens:", len(chunk_context) // 4)
    print("=" * 70)

    for _, row in chunk.iterrows():

        question_id = str(row["Question ID"])

        # =========================
        # ENGLISH
        # =========================

        prompt_en = f"""
You are answering a benchmark question.

Use ONLY the information contained in the DATASET below.

Rules:
- Answer the question directly.
- Do not invent information.
- Do not use outside information.
- Do not repeat the question.
- Give only the answer.
- Do not add explanations or summaries.
- If the answer cannot be determined from the dataset, respond exactly:
NOT FOUND

DATASET:
{chunk_context}

QUESTION:
{row["English Question"]}

ANSWER:
"""

        try:
            response_en = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_en,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if response_en.status_code == 200:
                answer_en = response_en.json().get(
                    "response", ""
                ).strip()
            else:
                answer_en = f"REQUEST ERROR {response_en.status_code}"

        except Exception as e:
            answer_en = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "English",
            "Question": row["English Question"],
            "LLM Answer": answer_en,
            "Dataset Answer": row["Gold Answer (EN)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | English | done")

        # =========================
        # NEPALI
        # =========================

        prompt_ne = f"""
तपाईं benchmark प्रश्नको उत्तर दिइरहनुभएको छ।

तलको DATASET मा भएको जानकारी मात्र प्रयोग गर्नुहोस्।

नियमहरू:
- प्रश्नको सिधा उत्तर दिनुहोस्।
- जानकारी नबनाउनुहोस्।
- बाहिरी जानकारी प्रयोग नगर्नुहोस्।
- प्रश्न दोहोर्याउनुहोस्।
- केवल उत्तर दिनुहोस्।
- explanation वा summary नदिनुहोस्।
- DATASET बाट उत्तर निर्धारण गर्न नसके:
NOT FOUND
भन्नुहोस्।

DATASET:
{chunk_context}

QUESTION:
{row["Nepali Question"]}

ANSWER:
"""

        try:
            response_ne = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt_ne,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            if response_ne.status_code == 200:
                answer_ne = response_ne.json().get(
                    "response", ""
                ).strip()
            else:
                answer_ne = f"REQUEST ERROR {response_ne.status_code}"

        except Exception as e:
            answer_ne = f"REQUEST ERROR: {e}"

        all_results.append({
            "Question ID": question_id,
            "Language": "Nepali",
            "Question": row["Nepali Question"],
            "LLM Answer": answer_ne,
            "Dataset Answer": row["Gold Answer (NE)"],
            "Manual Verdict": "",
            "Error Type": ""
        })

        print(f"{question_id} | Nepali | done")

    # Save progress after each chunk
    temp_file = os.path.join(
        folder,
        "Culture_Religion_Literature_Gemma3_1B_TEMP.csv"
    )

    pd.DataFrame(all_results).to_csv(
        temp_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("Temporary progress saved:", len(all_results), "responses")

print("\n" + "=" * 70)
print("GEMMA RUN COMPLETE")
print("=" * 70)

print("Total responses:", len(all_results))
print("Expected responses:", len(df) * 2)

Total questions: 424
Chunk size: 150
Number of chunks: 3

CHUNK 1/3
Rows 1 to 150
Context characters: 64769
Approximate tokens: 16192
CUL001 | English | done
CUL001 | Nepali | done
CUL002 | English | done
CUL002 | Nepali | done
CUL003 | English | done
CUL003 | Nepali | done
CUL004 | English | done
CUL004 | Nepali | done
CUL005 | English | done
CUL005 | Nepali | done
CUL006 | English | done
CUL006 | Nepali | done
CUL007 | English | done
CUL007 | Nepali | done
CUL008 | English | done
CUL008 | Nepali | done
CUL009 | English | done
CUL009 | Nepali | done
CUL010 | English | done


In [ ]:
results_df = pd.DataFrame(all_results)

output_file = os.path.join(
    folder,
    "Culture_Religion_Literature_Gemma3_1B_Results.csv"
)

results_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(output_file)

print("\nRows:", len(results_df))
print("Expected:", len(df) * 2)